# Steam Game Reviews -> Silver y DLQ

Conserva `reviews` como texto porque el diagnóstico no demuestra una estructura interna estable. El batch actual se limpia, se deduplica por `app_id` y se escribe en Silver; las filas inválidas se aíslan en DLQ.

In [ ]:
import json

dbutils.widgets.text("catalog", "steam_game_intelligence")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("silver_schema", "silver")
dbutils.widgets.text("dlq_schema", "dlq")
dbutils.widgets.text("cutoff_us", "")
dbutils.widgets.text("run_id", "")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
dlq_schema = dbutils.widgets.get("dlq_schema")
cutoff_us = dbutils.widgets.get("cutoff_us")
run_id = dbutils.widgets.get("run_id")
if not cutoff_us.isdigit():
    raise ValueError("cutoff_us debe ser un epoch en microsegundos")


In [ ]:
%sql
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.' || :silver_schema || '.steam_game_reviews') AS
WITH typed AS (
  SELECT
    try_cast(app_id AS BIGINT) AS app_id,
    trim(name) AS name,
    trim(reviews) AS reviews,
    _ingested_at,
    trim(_source) AS _source
  FROM IDENTIFIER(:catalog || '.' || :bronze_schema || '.steam_game_reviews_batch')
  WHERE _ingested_at <= timestamp_micros(try_cast(:cutoff_us AS BIGINT))
), ranked AS (
  SELECT *, row_number() OVER (
    PARTITION BY app_id
    ORDER BY _ingested_at DESC, sha2(to_json(struct(*)), 256) DESC
  ) AS rn
  FROM typed
), valid AS (
  SELECT *
  FROM ranked
  WHERE rn = 1
    AND app_id IS NOT NULL AND app_id > 0
    AND name IS NOT NULL AND name <> ''
    AND reviews IS NOT NULL AND reviews <> ''
    AND _ingested_at IS NOT NULL
    AND _source IS NOT NULL AND _source <> ''
)
SELECT app_id, name, reviews, _ingested_at, _source
FROM valid;

In [ ]:
%sql
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.' || :dlq_schema || '.steam_game_reviews_rejected') AS
WITH typed AS (
  SELECT
    try_cast(app_id AS BIGINT) AS app_id,
    name,
    reviews,
    _ingested_at,
    _source
  FROM IDENTIFIER(:catalog || '.' || :bronze_schema || '.steam_game_reviews_batch')
  WHERE _ingested_at <= timestamp_micros(try_cast(:cutoff_us AS BIGINT))
), ranked AS (
  SELECT *, row_number() OVER (
    PARTITION BY app_id
    ORDER BY _ingested_at DESC, sha2(to_json(struct(*)), 256) DESC
  ) AS rn
  FROM typed
), rejected AS (
  SELECT *, filter(array(
    CASE WHEN app_id IS NULL OR app_id <= 0 THEN 'app_id_invalido' END,
    CASE WHEN name IS NULL OR trim(name) = '' THEN 'name_vacio' END,
    CASE WHEN reviews IS NULL OR trim(reviews) = '' THEN 'reviews_vacio' END,
    CASE WHEN _ingested_at IS NULL THEN 'ingested_at_nulo' END,
    CASE WHEN _source IS NULL OR trim(_source) = '' THEN 'source_vacio' END
  ), x -> x IS NOT NULL) AS error_reasons
  FROM ranked
  WHERE rn = 1
)
SELECT
  sha2(concat_ws('|', coalesce(cast(app_id AS STRING), 'NULL'), cast(_ingested_at AS STRING), to_json(struct(*))), 256) AS _reject_key,
  app_id,
  array_join(error_reasons, '; ') AS motivo_rechazo,
  error_reasons AS reglas_violadas,
  to_json(struct(*)) AS payload,
  :run_id AS _run_id,
  current_timestamp() AS _rejected_at
FROM rejected
WHERE size(error_reasons) > 0;

In [ ]:
dbutils.notebook.exit(json.dumps({"status": "completed", "table": "steam_game_reviews"}))